# TVAE

Third of the four generation notebooks, following the same shape as the previous two.

TVAE takes a different neural approach from CTGAN. Rather than two networks competing, one
network compresses each record into a small internal summary and a second reconstructs
records from that summary. New records are generated by sampling fresh points in the
summary space and decoding them. There is no adversarial game, which generally makes it
more stable to train.

The trade-off, noted in the literature review, is that the compression step can smooth away
rare or unusual patterns, since those are the easiest thing to lose when records are
squeezed through a small summary. The fidelity checks on rare categories will show whether
that happened here.

## Setup

In [1]:
%pip install -q pandas pyarrow sdv

## Working folder

Sets the project folder so everything the pipeline writes, the cohort, the synthetic
datasets, the outputs and the figures, persists between sessions rather than sitting on
temporary storage.

The cohort and the synthetic datasets derive from MIMIC-IV, which is credentialed data
under a PhysioNet data use agreement. Keep the folder private, do not share it, and
delete the data once the work is finished.

In [2]:
import os
from pathlib import Path

# Use the shared project folder when one is available, otherwise stay in the
# current directory.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    project_dir = Path("/content/drive/MyDrive/mimic-synthetic-pipeline")
    project_dir.mkdir(parents=True, exist_ok=True)
    os.chdir(project_dir)
    print(f"Working folder: {project_dir}")
except ImportError:
    print("Using the local working folder.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working folder set to Google Drive: /content/drive/MyDrive/mimic-synthetic-pipeline


## Hardware check

TVAE is a neural network like CTGAN, so the same hardware note applies: minutes on a GPU,
potentially more than an hour on a CPU.

In [3]:
import torch

cuda_available = torch.cuda.is_available()
print(f"CUDA available: {cuda_available}")
if cuda_available:
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print(
        "No GPU detected. TVAE will train on CPU, which can take over an hour on this "
        "cohort. If a GPU was expected, enable it in the session settings, then "
        "restart the session and re-run from the beginning."
    )

CUDA available: True
GPU: NVIDIA L4


## Training data

In [4]:
from pathlib import Path

import pandas as pd

if not Path("data/train.parquet").exists():
    raise FileNotFoundError(
        "data/train.parquet not found. Run the extraction and preparation steps first."
    )

TARGET = "readmitted_30d"
train_df = pd.read_parquet("data/train.parquet")
print(f"Training data: {len(train_df):,} admissions, readmission rate {train_df[TARGET].mean():.4f}")

Training data: 427,408 admissions, readmission rate 0.2067


## Fitting and generating

Each method is trained to its own convergence rather than to a shared epoch count.
Matching epoch counts sounds fairer but is not, because an epoch means something different
for an adversarial game, a variational autoencoder and a diffusion model. Quality is
compared at each method's own ceiling, and the cost of reaching it is reported separately
in the timing log.

Unlike CTGAN, TVAE optimises a single reconstruction objective, so its loss curve is a
genuine convergence signal and the stopping rule can be applied properly. The budget is
set generously and the convergence check below reports whether the loss had flattened by
the end.

In [5]:
import time

from sdv.metadata import Metadata
from sdv.single_table import TVAESynthesizer

EPOCHS = 600  # Raised from 300, where the convergence check reported the loss still
              # improving by 1.95% across the final fifth, above the 1% threshold. At
              # roughly 7.3 seconds per epoch on an L4 this is about 73 minutes. If the
              # check still reports more than 1% at 600, raise it again and re-run.

metadata = Metadata.detect_from_dataframe(train_df, table_name="cohort")
model = TVAESynthesizer(metadata, epochs=EPOCHS, verbose=True)

t0 = time.time()
model.fit(train_df)
train_seconds = time.time() - t0

t0 = time.time()
synthetic_df = model.sample(num_rows=len(train_df))
generate_seconds = time.time() - t0

synthetic_df.to_parquet("data/synthetic_tvae.parquet", index=False)
print(f"Training took {train_seconds:.1f}s, generation took {generate_seconds:.1f}s")
print(f"Saved {len(synthetic_df):,} synthetic admissions to data/synthetic_tvae.parquet")

/usr/local/lib/python3.12/dist-packages/sdv/single_table/base.py:139: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
Loss: +03.38: 100%|██████████| 600/600 [1:08:41<00:00,  6.87s/it]


Training took 4223.3s, generation took 3.1s
Saved 427,408 synthetic admissions to data/synthetic_tvae.parquet


## Training convergence

TVAE optimises a single reconstruction objective with no adversarial component, so unlike
CTGAN its loss curve can be read as a convergence signal in the ordinary way. If the loss
is still falling appreciably at the last epoch, the model is undertrained.

The library records a loss value per batch, so values are averaged within each epoch to
give one point per epoch. Rather than eyeballing the shape, the cell compares the mean loss
over the final fifth of training against the fifth before it and prints the percentage
improvement, so the epoch count can be justified with a number.

In [ ]:
import csv

import matplotlib.pyplot as plt


def save_csv(df, path):
    """Write a dataframe to CSV, falling back to the standard library.

    Installing sdv can leave the session with a pandas whose CSV writer is broken
    ("AttributeError: 'Index' object has no attribute '_format_native_types'"),
    because pip replaces pandas on disk while the kernel still holds parts of the
    previous version. The stdlib writer does not touch pandas internals, so it works
    regardless. Reported either way so it is clear which path was taken.
    """
    try:
        df.to_csv(path, index=False)
        print(f"Saved {path}")
        return True
    except Exception as exc:  # noqa: BLE001
        try:
            with open(path, "w", newline="", encoding="utf-8") as fh:
                writer = csv.writer(fh)
                writer.writerow([str(c) for c in df.columns])
                writer.writerows(df.itertuples(index=False, name=None))
            print(f"Saved {path} (via the stdlib fallback; pandas raised {type(exc).__name__})")
            return True
        except Exception as exc2:  # noqa: BLE001
            print(f"Could not write {path}: {type(exc2).__name__}: {exc2}")
            return False


fig_dir = Path("figures")
fig_dir.mkdir(exist_ok=True)
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)

loss_df = model.get_loss_values()  # one row per batch
per_epoch = loss_df.groupby("Epoch")["Loss"].mean()

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(per_epoch.index, per_epoch.values, color="#55A868")
ax.set_xlabel("Epoch")
ax.set_ylabel("Mean loss")
ax.set_title(f"TVAE training convergence over {EPOCHS} epochs")
fig.tight_layout()
fig.savefig(fig_dir / "training_convergence_tvae.png", dpi=150)
plt.show()

# The convergence verdict is computed and printed BEFORE anything is written to disk,
# so a file-writing problem cannot cost the diagnostic after an hour of training.
n = len(per_epoch)
prev_fifth = per_epoch.iloc[int(n * 0.6):int(n * 0.8)].mean()
last_fifth = per_epoch.iloc[int(n * 0.8):].mean()
improvement = (prev_fifth - last_fifth) / abs(prev_fifth) * 100

print(f"Mean loss, epochs {int(n * 0.6) + 1}-{int(n * 0.8)}: {prev_fifth:.4f}")
print(f"Mean loss, final {n - int(n * 0.8)} epochs: {last_fifth:.4f}")
print(f"Improvement across the final fifth: {improvement:+.2f}%")

if improvement > 1.0:
    print(
        f"\nStill improving by more than 1% across the final fifth of training at "
        f"EPOCHS={EPOCHS}. Note the 1% figure is a stopping heuristic, not a law: weigh "
        f"it against the absolute change in the loss and the training time before "
        f"raising the budget again."
    )
else:
    print(
        f"\nFlat to within 1% across the final fifth, which is consistent with the "
        f"model having converged at EPOCHS={EPOCHS}."
    )

print()
save_csv(loss_df, out_dir / "loss_history_tvae.csv")

## Sanity checks

The same quick aggregate checks as the other generation notebooks.

In [11]:
checks = pd.DataFrame({
    "statistic": ["Readmission rate", "Mean age", "Mean length of stay (days)"],
    "real_training_data": [
        round(train_df[TARGET].mean(), 4),
        round(train_df["age_at_admission"].astype(float).mean(), 1),
        round(train_df["length_of_stay_days"].mean(), 2),
    ],
    "synthetic_data": [
        round(synthetic_df[TARGET].mean(), 4),
        round(synthetic_df["age_at_admission"].astype(float).mean(), 1),
        round(synthetic_df["length_of_stay_days"].mean(), 2),
    ],
})
checks

/usr/local/lib/python3.12/dist-packages/google/colab/_interactive_table_hint_button.py:178: FutureWarning: Index.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  df_html=dataframe._repr_html_(),  # pylint: disable=protected-access
/usr/local/lib/python3.12/dist-packages/google/colab/_interactive_table_hint_button.py:178: FutureWarning: RangeIndex.format is deprecated and will be removed in a future version. Convert using index.astype(str) or index.map(formatter) instead.
  df_html=dataframe._repr_html_(),  # pylint: disable=protected-access


,statistic,real_training_data,synthetic_data
0,Readmission rate,0.2067,0.2026
1,Mean age,58.8000,59.1000
2,Mean length of stay (days),4.6400,4.4500


In [ ]:
# Append this run's timings to the shared generation log used by all four methods.
out_dir = Path("outputs")
out_dir.mkdir(exist_ok=True)
log_path = out_dir / "generation_log.csv"

entry = pd.DataFrame([{
    "method": "TVAE",
    "rows_generated": len(synthetic_df),
    "train_seconds": round(train_seconds, 1),
    "generate_seconds": round(generate_seconds, 1),
}])
if log_path.exists():
    log = pd.read_csv(log_path)
    log = log[log["method"] != "TVAE"]
    log = pd.concat([log, entry], ignore_index=True)
else:
    log = entry

# save_csv is defined in the convergence cell above and falls back to the standard
# library, because installing sdv can leave pandas' CSV writer broken for the rest of
# the session. This log is a result rather than a convenience, so it must not be lost
# to that. Run the convergence cell first if save_csv is not yet defined.
save_csv(log, log_path)
log